# Phase III Reynolds Solution: Re = 1000, 33 x 33 (Standard Coupled SGS)

This notebook runs a single high-Reynolds-number square cavity case using the standard coupled SGS solver path:

| Reynolds number | Nodes | Relative mesh spacing label |
|---:|---:|---:|
| 1000 | `33 x 33` | `h = 4` |

It keeps `solver_method = 'coupled'` and does not switch to `fractional_step`. The solver is patched in memory, so `main_solver.py` is not modified. Results are written to a separate output folder from the longer `65 x 65` run.

In [1]:
from pathlib import Path
import contextlib
import io
import json
import os
import shutil
import time

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if Path.cwd().name != "start-code" and (Path.cwd() / "start-code").exists():
    os.chdir(Path.cwd() / "start-code")

plot_dir = Path("Phase III Reynolds Re1000 33x33 Coupled Standard")
plot_dir.mkdir(exist_ok=True)

cache_dir = plot_dir / "case_cache"
cache_dir.mkdir(exist_ok=True)

print(f"Working directory: {Path.cwd()}")
print(f"Plot output directory: {plot_dir.resolve()}")
print(f"Cache directory: {cache_dir.resolve()}")

Working directory: /Users/windy/mae4100-courseproject2/start-code
Plot output directory: /Users/windy/mae4100-courseproject2/start-code/Phase III Reynolds Re1000 33x33 Coupled Standard
Cache directory: /Users/windy/mae4100-courseproject2/start-code/Phase III Reynolds Re1000 33x33 Coupled Standard/case_cache


In [2]:
REYNOLDS_CASES = [
    {"Re": 1000.0, "nodes": 33, "h_label": 4, "cfl": 0.10},
]

# Keep cached results once they have been generated. Set FORCE_RERUN = True to recompute the case.
USE_CACHE = True
FORCE_RERUN = False

pd.DataFrame(REYNOLDS_CASES)

,Re,nodes,h_label,cfl
0,1000.0,33,4,0.1


In [3]:
MAIN_PLOT_FILES = [
    "ucontour.png",
    "vcontour.png",
    "pcontour.png",
    "residualcomponent.png",
    "residual.png",
]


def case_label(case):
    return f"Re{int(case['Re']):04d}_{case['nodes']}x{case['nodes']}"


def format_solver_float(value):
    return f"{value:<16g}"


def apply_replacements(source, replacements, solver_path):
    for old, new in replacements.items():
        if old not in source:
            raise RuntimeError(f"Could not find expected setting in {solver_path}: {old}")
        source = source.replace(old, new, 1)
    return source


def patched_solver_source(case):
    solver_path = Path("main_solver.py")
    source = solver_path.read_text()
    nodes = case["nodes"]

    replacements = {
        "imax = 9               # Number of points in the x-direction (use odd numbers only)":
            f"imax = {nodes:<15}# Number of points in the x-direction (use odd numbers only)",
        "jmax = 9               # Number of points in the y-direction (use odd numbers only)":
            f"jmax = {nodes:<15}# Number of points in the y-direction (use odd numbers only)",
        "cfl = 0.5              # CFL number used to determine time step":
            f"cfl = {format_solver_float(case['cfl'])}# CFL number used to determine time step",
        "Re = 10.0              # Reynolds number = rho*Uinf*L/rmu":
            f"Re = {format_solver_float(case['Re'])}# Reynolds number = rho*Uinf*L/rmu",
        "iterout = 5000         # Number of time steps between solution output":
            "iterout = 100000000    # Number of time steps between solution output",
        "vectorize = False":
            "vectorize = True",
    }
    return solver_path, apply_replacements(source, replacements, solver_path)


def archive_solver_plots(label):
    case_plot_dir = plot_dir / label
    case_plot_dir.mkdir(exist_ok=True)
    for filename in MAIN_PLOT_FILES:
        src = Path(filename)
        if src.exists():
            dst = case_plot_dir / filename
            if dst.exists():
                dst.unlink()
            shutil.move(str(src), str(dst))


def save_case_cache(case, result):
    label = case_label(case)
    np.savez_compressed(
        cache_dir / f"{label}.npz",
        u=result["u"],
        convVector=result["convVector"],
        res=result["res"],
    )
    metadata = {k: v for k, v in result.items() if k not in {"u", "convVector", "res", "captured_output"}}
    metadata["captured_output_tail"] = result["captured_output"].splitlines()[-12:]
    (cache_dir / f"{label}.json").write_text(json.dumps(metadata, indent=2))


def load_case_cache(case):
    label = case_label(case)
    npz_path = cache_dir / f"{label}.npz"
    json_path = cache_dir / f"{label}.json"
    if not npz_path.exists() or not json_path.exists():
        return None

    arrays = np.load(npz_path)
    metadata = json.loads(json_path.read_text())
    metadata["u"] = arrays["u"]
    metadata["convVector"] = arrays["convVector"]
    metadata["res"] = arrays["res"]
    metadata["captured_output"] = "\n".join(metadata.pop("captured_output_tail", []))
    return metadata


def run_solver_case(case):
    label = case_label(case)
    cached = load_case_cache(case) if USE_CACHE and not FORCE_RERUN else None
    if cached is not None:
        print(f"Loaded cached result for {label}")
        return cached

    solver_path, source = patched_solver_source(case)
    namespace = {
        "__file__": str(solver_path.resolve()),
        "__name__": "__main__",
    }

    print(
        f"Running coupled SGS {label}: "
        f"Re = {case['Re']:g}, nodes = {case['nodes']}x{case['nodes']}, "
        f"h = {case['h_label']}, CFL = {case['cfl']}"
    )
    start_time = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()) as captured_output:
        exec(compile(source, str(solver_path), "exec"), namespace)
    elapsed_time = time.perf_counter() - start_time
    archive_solver_plots(label)
    plt.close("all")

    u = namespace["u"].copy()
    imax, jmax, _ = u.shape
    result = {
        "label": label,
        "solver_method": namespace["solver_method"],
        "Re": case["Re"],
        "nodes": case["nodes"],
        "h_label": case["h_label"],
        "cfl": case["cfl"],
        "dx": (namespace["xmax"] - namespace["xmin"]) / (imax - 1),
        "dy": (namespace["ymax"] - namespace["ymin"]) / (jmax - 1),
        "viscosity": float(namespace["rmu"]),
        "iterations": int(namespace["n"]),
        "converged": bool(namespace["isConverged"]),
        "final_conv": float(namespace["conv"]),
        "elapsed_time_sec": float(elapsed_time),
        "xmin": float(namespace["xmin"]),
        "xmax": float(namespace["xmax"]),
        "ymin": float(namespace["ymin"]),
        "ymax": float(namespace["ymax"]),
        "u": u,
        "res": np.array(namespace["res"], copy=True),
        "convVector": np.array(namespace["convVector"], copy=True),
        "captured_output": captured_output.getvalue(),
    }
    save_case_cache(case, result)
    return result


def build_summary(results):
    return pd.DataFrame([
        {
            "Re": result["Re"],
            "nodes": f"{result['nodes']}x{result['nodes']}",
            "solver method": result["solver_method"],
            "h label": result["h_label"],
            "dx = dy (m)": result["dx"],
            "CFL": result["cfl"],
            "viscosity (N*s/m^2)": result["viscosity"],
            "iterations": result["iterations"],
            "converged": result["converged"],
            "final conv": result["final_conv"],
            "wall time (s)": result["elapsed_time_sec"],
        }
        for result in results
    ])


def write_partial_summary(results):
    summary = build_summary(results)
    summary.to_csv(plot_dir / "summary.csv", index=False)
    summary.to_csv(cache_dir / "summary.csv", index=False)
    return summary

In [4]:
results = []
for case in REYNOLDS_CASES:
    result = run_solver_case(case)
    results.append(result)
    summary = write_partial_summary(results)
    print(f"Saved summary: {plot_dir / 'summary.csv'}")

summary

Running coupled SGS Re1000_33x33: Re = 1000, nodes = 33x33, h = 4, CFL = 0.1


/Users/windy/mae4100-courseproject2/start-code/req_functions.py:546: RuntimeWarning: overflow encountered in scalar divide
  d4pdy4 = (u[i, j - 2, 0] - 4.0 * u[i, j - 1, 0] + 6.0 * u[i, j, 0] - 4.0 * u[i, j + 1, 0] + u[i, j + 2, 0]) / (dy ** 4)
/Users/windy/mae4100-courseproject2/start-code/req_functions.py:546: RuntimeWarning: invalid value encountered in scalar add
  d4pdy4 = (u[i, j - 2, 0] - 4.0 * u[i, j - 1, 0] + 6.0 * u[i, j, 0] - 4.0 * u[i, j + 1, 0] + u[i, j + 2, 0]) / (dy ** 4)
/Users/windy/mae4100-courseproject2/start-code/req_functions.py:550: RuntimeWarning: invalid value encountered in scalar subtract
  ymom_rhs = -rho * (uvel * dvdx + vvel * dvdy) - dpdy + rmu * (d2vdx2 + d2vdy2) + s[i, j, 2]
/Users/windy/mae4100-courseproject2/start-code/req_functions.py:544: RuntimeWarning: overflow encountered in scalar divide
  d4pdx4 = (u[i - 2, j, 0] - 4.0 * u[i - 1, j, 0] + 6.0 * u[i, j, 0] - 4.0 * u[i + 1, j, 0] + u[i + 2, j, 0]) / (dx ** 4)
/Users/windy/mae4100-courseproject2/sta

KeyboardInterrupt: 

In [ ]:
summary_csv = plot_dir / "summary.csv"
if not summary_csv.exists():
    raise FileNotFoundError("No summary found yet. Run the case loop first.")

print(f"Loaded summary from: {summary_csv.resolve()}")
summary = pd.read_csv(summary_csv)
display(summary)

In [ ]:
def normalized_coordinates(result):
    x = np.linspace(result["xmin"], result["xmax"], result["nodes"])
    y = np.linspace(result["ymin"], result["ymax"], result["nodes"])
    length = result["xmax"] - result["xmin"]
    return x / length, y / length


fontsize = 12
result = results[0]
x_norm, y_norm = normalized_coordinates(result)
u = result["u"]
i_mid = (result["nodes"] - 1) // 2
j_mid = (result["nodes"] - 1) // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)

axes[0].plot(u[i_mid, :, 1], y_norm, linewidth=2)
axes[0].set_xlabel("u velocity (m/s)", fontsize=fontsize)
axes[0].set_ylabel("y / L", fontsize=fontsize)
axes[0].set_title("Vertical centerline u-velocity", fontsize=fontsize)
axes[0].grid(True, alpha=0.25)

axes[1].plot(x_norm, u[:, j_mid, 2], linewidth=2)
axes[1].set_xlabel("x / L", fontsize=fontsize)
axes[1].set_ylabel("v velocity (m/s)", fontsize=fontsize)
axes[1].set_title("Horizontal centerline v-velocity", fontsize=fontsize)
axes[1].grid(True, alpha=0.25)

axes[2].plot(u[i_mid, :, 0], y_norm, linewidth=2)
axes[2].set_xlabel("p (N/m^2)", fontsize=fontsize)
axes[2].set_ylabel("y / L", fontsize=fontsize)
axes[2].set_title("Vertical centerline pressure", fontsize=fontsize)
axes[2].grid(True, alpha=0.25)

fig.suptitle("Re = 1000, 33 x 33, Standard Coupled SGS")
fig.savefig(plot_dir / "centerline_profiles_Re1000_33x33.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
X, Y = np.meshgrid(x_norm, y_norm, indexing="ij")
fields = [
    (u[:, :, 1], "u velocity (m/s)", "u_velocity_contour_Re1000_33x33.png"),
    (u[:, :, 2], "v velocity (m/s)", "v_velocity_contour_Re1000_33x33.png"),
    (u[:, :, 0], "pressure (N/m^2)", "pressure_contour_Re1000_33x33.png"),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3), constrained_layout=True)
for ax, (field, title, _) in zip(axes, fields):
    contour = ax.contourf(X, Y, field, levels=24)
    fig.colorbar(contour, ax=ax)
    ax.set_aspect("equal")
    ax.set_xlabel("x / L")
    ax.set_ylabel("y / L")
    ax.set_title(title)

fig.suptitle("Re = 1000, 33 x 33, Standard Coupled SGS")
fig.savefig(plot_dir / "field_contours_Re1000_33x33.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
flow_metrics = pd.DataFrame([
    {
        "Re": result["Re"],
        "nodes": f"{result['nodes']}x{result['nodes']}",
        "solver_method": result["solver_method"],
        "u_min": np.min(u[:, :, 1]),
        "u_max": np.max(u[:, :, 1]),
        "v_min": np.min(u[:, :, 2]),
        "v_max": np.max(u[:, :, 2]),
        "p_min": np.min(u[:, :, 0]),
        "p_max": np.max(u[:, :, 0]),
    }
])
flow_metrics.to_csv(plot_dir / "flow_metrics_Re1000_33x33.csv", index=False)
flow_metrics